In [ ]:
import MetaTrader5 as mt5
import pandas as pd
import time
from datetime import datetime, timedelta

# --- ⚙️ EXPERT CONFIGURATION ---
SYMBOL = "XAUUSD"          # TRADING GOLD
TIMEFRAME = mt5.TIMEFRAME_H1  # HOURLY CHARTS
LOT_SIZE = 0.10            # 0.10 Lots ($10 per $1 move)
MAGIC_NUMBER = 20260206    # Unique ID
DAILY_LOSS_LIMIT = -1000   # 🛡️ SAFETY NET

# --- 🎯 TARGET SETTINGS (THE FIX) ---
SL_DOLLARS = 12.0          # Stop Loss distance in Price ($12 move = $120 Risk)
TP_DOLLARS = 36.0          # Take Profit distance in Price ($36 move = $360 Profit)

# --- 🔌 CONNECT ---
if not mt5.initialize():
    print(f"❌ FAILED to connect to MT5. Error: {mt5.last_error()}")
    quit()

# GET ACCOUNT INFO
account = mt5.account_info()
print(f"🤖 DEPLOYING: GOLD MINER PRO (FIXED TP/SL)")
print(f"   Account: {account.login} | Balance: ${account.balance:.2f}")
print(f"   SL: ${SL_DOLLARS} price move | TP: ${TP_DOLLARS} price move")
print(f"-------------------------------------------------------")

def get_daily_pnl():
    """Calculate Profit/Loss for the current day"""
    history_orders = mt5.history_deals_get(
        (datetime.now() - timedelta(hours=24)), 
        datetime.now()
    )
    if history_orders is None: return 0.0
    
    daily_profit = 0.0
    for deal in history_orders:
        if deal.magic == MAGIC_NUMBER and deal.entry == mt5.DEAL_ENTRY_OUT:
            daily_profit += deal.profit + deal.swap + deal.commission
            
    return daily_profit

def get_data():
    """Fetch H1 Data & Calculate EMA"""
    rates = mt5.copy_rates_from_pos(SYMBOL, TIMEFRAME, 0, 100)
    if rates is None: return None
    df = pd.DataFrame(rates)
    
    # EMA 10 (Fast) and EMA 20 (Slow)
    df['EMA_FAST'] = df['close'].ewm(span=10, adjust=False).mean()
    df['EMA_SLOW'] = df['close'].ewm(span=20, adjust=False).mean()
    return df

def close_positions(direction):
    """Close trades of the opposite direction"""
    positions = mt5.positions_get(symbol=SYMBOL)
    if positions:
        for pos in positions:
            type_to_close = 1 if direction == "BUY" else 0
            
            if pos.type == type_to_close and pos.magic == MAGIC_NUMBER:
                req = {
                    "action": mt5.TRADE_ACTION_DEAL,
                    "position": pos.ticket,
                    "symbol": SYMBOL,
                    "volume": pos.volume,
                    "type": mt5.ORDER_TYPE_BUY if pos.type == 1 else mt5.ORDER_TYPE_SELL,
                    "price": mt5.symbol_info_tick(SYMBOL).ask if pos.type == 1 else mt5.symbol_info_tick(SYMBOL).bid,
                    "magic": MAGIC_NUMBER,
                    "comment": "Trend Flip Close",
                    "type_time": mt5.ORDER_TIME_GTC,
                    "type_filling": mt5.ORDER_FILLING_IOC,
                }
                mt5.order_send(req)
                print(f"🔄 CLOSED Opposing Trade to flip direction.")

def open_trade(direction):
    """Open new trade with FIXED TP and SL"""
    # 1. Check Safety Net
    pnl = get_daily_pnl()
    if pnl <= DAILY_LOSS_LIMIT:
        print(f"🛑 DAILY LIMIT HIT (${pnl:.2f}). Trading Paused.")
        return

    # 2. Check Existing Trades
    positions = mt5.positions_get(symbol=SYMBOL)
    for pos in positions:
        if pos.magic == MAGIC_NUMBER:
            if (direction == "BUY" and pos.type == 0) or (direction == "SELL" and pos.type == 1):
                return # Already in the correct trade
            
    close_positions(direction)
    
    # 3. Execute New Trade
    tick = mt5.symbol_info_tick(SYMBOL)
    price = tick.ask if direction == "BUY" else tick.bid
    type_order = mt5.ORDER_TYPE_BUY if direction == "BUY" else mt5.ORDER_TYPE_SELL
    
    # --- 🛠️ THE FIX IS HERE ---
    if direction == "BUY":
        sl_price = price - SL_DOLLARS
        tp_price = price + TP_DOLLARS
    else:
        sl_price = price + SL_DOLLARS
        tp_price = price - TP_DOLLARS
    
    req = {
        "action": mt5.TRADE_ACTION_DEAL,
        "symbol": SYMBOL,
        "volume": LOT_SIZE,
        "type": type_order,
        "price": price,
        "sl": sl_price,
        "tp": tp_price,  # ✅ ADDED TAKE PROFIT
        "magic": MAGIC_NUMBER,
        "comment": "Gold Miner Pro",
        "type_time": mt5.ORDER_TIME_GTC,
        "type_filling": mt5.ORDER_FILLING_IOC,
    }
    
    res = mt5.order_send(req)
    if res.retcode == mt5.TRADE_RETCODE_DONE:
        print(f"🚀 OPENED {direction} @ {price}")
        print(f"   🛑 SL: {sl_price} (-$120 approx)")
        print(f"   🎯 TP: {tp_price} (+$360 approx)")
    else:
        print(f"❌ ERROR Opening {direction}: {res.comment}")

# --- 🔁 MAIN LOOP ---
print("⏳ SCANNING H1 TREND (Updates every 10 seconds)...")

while True:
    try:
        current_daily_pnl = get_daily_pnl()
        if current_daily_pnl <= DAILY_LOSS_LIMIT:
            print(f"🛑 DAILY LOSS LIMIT REACHED (${current_daily_pnl:.2f}). SLEEPING...", end='\r')
            time.sleep(60)
            continue

        df = get_data()
        if df is not None:
            curr = df.iloc[-2]
            prev = df.iloc[-3]
            
            cross_up = (prev['EMA_FAST'] <= prev['EMA_SLOW']) and (curr['EMA_FAST'] > curr['EMA_SLOW'])
            cross_down = (prev['EMA_FAST'] >= prev['EMA_SLOW']) and (curr['EMA_FAST'] < curr['EMA_SLOW'])
            
            # Display Status
            last_price = df.iloc[-1]['close']
            print(f"Scanning... PnL Today: ${current_daily_pnl:.2f} | Gold Price: {last_price:.2f}", end='\r')
            
            if cross_up:
                print("\n⚡ BUY SIGNAL DETECTED (Golden Cross)")
                open_trade("BUY")
            elif cross_down:
                print("\n⚡ SELL SIGNAL DETECTED (Death Cross)")
                open_trade("SELL")
                
        time.sleep(10)
        
    except Exception as e:
        print(f"⚠️ Error: {e}")
        time.sleep(5)

In [6]:
import MetaTrader5 as mt5
import pandas as pd
import pandas_ta as ta
import numpy as np
import random
from datetime import datetime, timedelta
import warnings

warnings.filterwarnings('ignore')

# --- PROP FIRM PARAMETERS ---
STARTING_CAPITAL = 25000.0
RISK_PER_TRADE_PCT = 0.005     # 0.5% Risk ($125)
MAX_TOTAL_DD = 0.10            # 10% Max Total Drawdown ($2,500)
PROFIT_TARGET = 0.08           # 8% Target ($2,000)
SPREAD_COMMISSION = 5.0        # $5 per lot round trip

def run_zscore_backtest():
    print("🔄 Connecting to MT5 to fetch Institutional Z-Score Data...")
    if not mt5.initialize():
        print("❌ MT5 Initialization Failed.")
        return
        
    # Fetch 2 Years of data
    date_to = datetime.now()
    date_from = date_to - timedelta(days=730) 
    
    rates = mt5.copy_rates_range("XAUUSD", mt5.TIMEFRAME_H1, date_from, date_to)
    mt5.shutdown()
    
    if rates is None or len(rates) == 0:
        print("❌ Failed to fetch data.")
        return
        
    df = pd.DataFrame(rates)
    df['time'] = pd.to_datetime(df['time'], unit='s')
    df.set_index('time', inplace=True)
    
    # ==========================================
    # 1. Z-SCORE & INSTITUTIONAL INDICATORS
    # ==========================================
    # ATR for Dynamic Risk
    df['atr'] = ta.atr(df['high'], df['low'], df['close'], length=14)
    
    # Macro Trend
    df['ema_200'] = ta.ema(df['close'], length=200)
    
    # Z-Score Calculation (20-period moving average & standard deviation)
    df['sma_20'] = ta.sma(df['close'], length=20)
    df['std_20'] = df['close'].rolling(window=20).std()
    df['z_score'] = (df['close'] - df['sma_20']) / df['std_20']
    
    df.dropna(inplace=True)
    
    # ==========================================
    # 2. THE Z-SCORE "DIP-BUYING" ALGORITHM
    # ==========================================
    # Power Hours: 13:00 to 18:00 UTC (NY Trend)
    ALLOWED_HOURS = [13, 14, 15, 16, 17, 18]
    trades = []
    
    for i in range(1, len(df) - 48):
        hour = df.index[i].hour
        
        if hour in ALLOWED_HOURS:
            # --- BUY SETUPS (Macro Trend UP, Z-Score Oversold) ---
            if df['close'].iloc[i] > df['ema_200'].iloc[i]: # Trend is UP
                if df['z_score'].iloc[i] < -1.5: # Price dropped 1.5 StdDevs (Stop Hunt)
                    entry = df['close'].iloc[i]
                    sl_dist = df['atr'].iloc[i] * 1.5  # Dynamic Stop
                    tp_dist = df['atr'].iloc[i] * 2.5  # Dynamic Target (1:1.66 R:R)
                    trades.append(simulate_trade_with_breakeven(df, i, entry, sl_dist, tp_dist, "BUY"))
                    
            # --- SELL SETUPS (Macro Trend DOWN, Z-Score Overbought) ---
            elif df['close'].iloc[i] < df['ema_200'].iloc[i]: # Trend is DOWN
                if df['z_score'].iloc[i] > 1.5: # Price spiked 1.5 StdDevs (Stop Hunt)
                    entry = df['close'].iloc[i]
                    sl_dist = df['atr'].iloc[i] * 1.5
                    tp_dist = df['atr'].iloc[i] * 2.5
                    trades.append(simulate_trade_with_breakeven(df, i, entry, sl_dist, tp_dist, "SELL"))

    trades = [t for t in trades if t is not None]
    print(f"✅ Found {len(trades)} valid Z-Score Sniper setups in 2 Years.")
    
    # ==========================================
    # 3. MONTE CARLO STRESS TEST
    # ==========================================
    print("\n" + "="*70)
    print("🏆 THE Z-SCORE V6: $25,000 PROP FIRM STRESS TEST (w/ Breakeven Logic)")
    print("="*70)
    
    pass_count = 0
    fail_count = 0
    
    for _ in range(1000):
        shuffled_trades = trades.copy()
        random.shuffle(shuffled_trades)
        
        mc_balance = STARTING_CAPITAL
        mc_peak = STARTING_CAPITAL
        
        for outcome in shuffled_trades:
            risk_amount = mc_balance * RISK_PER_TRADE_PCT
            sl_dollars = outcome['sl_dist']
            tp_dollars = outcome['tp_dist']
            
            lot_size = round(risk_amount / (sl_dollars * 100), 2)
            if lot_size <= 0: lot_size = 0.01
            
            if outcome['result'] == "WIN":
                mc_balance += (tp_dollars * 100 * lot_size) - (SPREAD_COMMISSION * lot_size)
            elif outcome['result'] == "LOSS":
                mc_balance -= (sl_dollars * 100 * lot_size) + (SPREAD_COMMISSION * lot_size)
            elif outcome['result'] == "BREAKEVEN":
                # Only pay spread/commission, zero loss on the asset price
                mc_balance -= (SPREAD_COMMISSION * lot_size)
                
            if mc_balance > mc_peak:
                mc_peak = mc_balance
                
            dd = (mc_peak - mc_balance) / mc_peak
            if dd >= MAX_TOTAL_DD:
                fail_count += 1
                break
            if mc_balance >= STARTING_CAPITAL * (1 + PROFIT_TARGET):
                pass_count += 1
                break

    mc_win_rate = (pass_count / 1000) * 100
    mc_fail_rate = (fail_count / 1000) * 100
    
    print(f"✅ Probability of PASSING Challenge: {mc_win_rate:.2f}%")
    print(f"❌ Probability of BLOWING Account:  {mc_fail_rate:.2f}%")
    print(f"🔄 Probability of running out of setups before passing: {100 - (mc_win_rate + mc_fail_rate):.2f}%")
    print("="*70)

def simulate_trade_with_breakeven(df, start_idx, entry, sl_dist, tp_dist, direction):
    """Simulates trade execution with Institutional Breakeven logic."""
    forward_df = df.iloc[start_idx+1 : start_idx+48] 
    
    # If the trade moves 1.0x ATR in our favor, we move Stop Loss to Breakeven
    breakeven_trigger_dist = sl_dist / 1.5 
    sl_moved_to_be = False
    
    for _, row in forward_df.iterrows():
        if direction == "BUY":
            # 1. Check if we hit the Breakeven Trigger (Price goes up)
            if not sl_moved_to_be and row['high'] >= entry + breakeven_trigger_dist:
                sl_moved_to_be = True
            
            # 2. Check if we hit Stop Loss
            if row['low'] <= entry - sl_dist:
                if sl_moved_to_be:
                    return {'result': 'BREAKEVEN', 'sl_dist': sl_dist, 'tp_dist': tp_dist}
                else:
                    return {'result': 'LOSS', 'sl_dist': sl_dist, 'tp_dist': tp_dist}
                    
            # 3. Check if we hit Full Take Profit
            if row['high'] >= entry + tp_dist:
                return {'result': 'WIN', 'sl_dist': sl_dist, 'tp_dist': tp_dist}
                
        else: # SELL
            # 1. Check if we hit the Breakeven Trigger (Price goes down)
            if not sl_moved_to_be and row['low'] <= entry - breakeven_trigger_dist:
                sl_moved_to_be = True
                
            # 2. Check if we hit Stop Loss
            if row['high'] >= entry + sl_dist:
                if sl_moved_to_be:
                    return {'result': 'BREAKEVEN', 'sl_dist': sl_dist, 'tp_dist': tp_dist}
                else:
                    return {'result': 'LOSS', 'sl_dist': sl_dist, 'tp_dist': tp_dist}
                    
            # 3. Check if we hit Full Take Profit
            if row['low'] <= entry - tp_dist:
                return {'result': 'WIN', 'sl_dist': sl_dist, 'tp_dist': tp_dist}
                
    return {'result': 'BREAKEVEN', 'sl_dist': sl_dist, 'tp_dist': tp_dist} # Expired in profit/chop

if __name__ == "__main__":
    run_zscore_backtest()

🔄 Connecting to MT5 to fetch Institutional Z-Score Data...
✅ Found 269 valid Z-Score Sniper setups in 2 Years.

🏆 THE Z-SCORE V6: $25,000 PROP FIRM STRESS TEST (w/ Breakeven Logic)
✅ Probability of PASSING Challenge: 100.00%
❌ Probability of BLOWING Account:  0.00%
🔄 Probability of running out of setups before passing: 0.00%


In [7]:
import MetaTrader5 as mt5
import pandas as pd
import pandas_ta as ta
import numpy as np
from datetime import datetime, timedelta
import warnings

warnings.filterwarnings('ignore')

# --- FUNDED ACCOUNT PARAMETERS ---
STARTING_CAPITAL = 25000.0
RISK_PER_TRADE_PCT = 0.005     # 0.5% Compounding Risk
SPREAD_COMMISSION = 5.0        # $5 per lot round trip

def run_funded_scaleup_backtest():
    print("🔄 Connecting to MT5 to fetch 1-Year XAUUSD Data...")
    if not mt5.initialize():
        print("❌ MT5 Initialization Failed.")
        return
        
    # Fetch Exactly 1 Year of Data
    date_to = datetime.now()
    date_from = date_to - timedelta(days=365) 
    
    rates = mt5.copy_rates_range("XAUUSD", mt5.TIMEFRAME_H1, date_from, date_to)
    mt5.shutdown()
    
    if rates is None or len(rates) == 0:
        print("❌ Failed to fetch data.")
        return
        
    df = pd.DataFrame(rates)
    df['time'] = pd.to_datetime(df['time'], unit='s')
    df.set_index('time', inplace=True)
    
    # ==========================================
    # 1. Z-SCORE & INSTITUTIONAL INDICATORS
    # ==========================================
    df['atr'] = ta.atr(df['high'], df['low'], df['close'], length=14)
    df['ema_200'] = ta.ema(df['close'], length=200)
    
    df['sma_20'] = ta.sma(df['close'], length=20)
    df['std_20'] = df['close'].rolling(window=20).std()
    df['z_score'] = (df['close'] - df['sma_20']) / df['std_20']
    
    df.dropna(inplace=True)
    
    # ==========================================
    # 2. EXTRACT CHRONOLOGICAL TRADES
    # ==========================================
    ALLOWED_HOURS = [13, 14, 15, 16, 17, 18]
    trades = []
    
    for i in range(1, len(df) - 48):
        hour = df.index[i].hour
        
        if hour in ALLOWED_HOURS:
            # BUY SETUPS
            if df['close'].iloc[i] > df['ema_200'].iloc[i] and df['z_score'].iloc[i] < -1.5:
                entry = df['close'].iloc[i]
                sl_dist = df['atr'].iloc[i] * 1.5
                tp_dist = df['atr'].iloc[i] * 2.5
                trade_date = df.index[i]
                result = simulate_trade_with_breakeven(df, i, entry, sl_dist, tp_dist, "BUY")
                trades.append({'date': trade_date, 'direction': 'BUY', **result})
                    
            # SELL SETUPS
            elif df['close'].iloc[i] < df['ema_200'].iloc[i] and df['z_score'].iloc[i] > 1.5:
                entry = df['close'].iloc[i]
                sl_dist = df['atr'].iloc[i] * 1.5
                tp_dist = df['atr'].iloc[i] * 2.5
                trade_date = df.index[i]
                result = simulate_trade_with_breakeven(df, i, entry, sl_dist, tp_dist, "SELL")
                trades.append({'date': trade_date, 'direction': 'SELL', **result})

    # ==========================================
    # 3. CHRONOLOGICAL COMPOUNDING SIMULATION
    # ==========================================
    print("\n" + "="*75)
    print("📈 THE FUNDED SCALER: 1-YEAR CHRONOLOGICAL COMPOUNDING REPORT")
    print("="*75)
    
    account_balance = STARTING_CAPITAL
    peak_balance = STARTING_CAPITAL
    max_drawdown = 0.0
    
    wins = 0
    losses = 0
    breakevens = 0
    
    monthly_performance = {}

    for t in trades:
        month_key = t['date'].strftime('%Y-%b')
        if month_key not in monthly_performance:
            monthly_performance[month_key] = {'start_bal': account_balance, 'end_bal': 0, 'trades': 0}
            
        monthly_performance[month_key]['trades'] += 1
        
        # Dynamic Compounding: Risk 0.5% of CURRENT balance
        risk_amount = account_balance * RISK_PER_TRADE_PCT
        sl_dollars = t['sl_dist']
        tp_dollars = t['tp_dist']
        
        lot_size = round(risk_amount / (sl_dollars * 100), 2)
        if lot_size <= 0: lot_size = 0.01
        
        if t['result'] == "WIN":
            account_balance += (tp_dollars * 100 * lot_size) - (SPREAD_COMMISSION * lot_size)
            wins += 1
        elif t['result'] == "LOSS":
            account_balance -= (sl_dollars * 100 * lot_size) + (SPREAD_COMMISSION * lot_size)
            losses += 1
        elif t['result'] == "BREAKEVEN":
            account_balance -= (SPREAD_COMMISSION * lot_size)
            breakevens += 1
            
        monthly_performance[month_key]['end_bal'] = account_balance
        
        # Track Drawdown
        if account_balance > peak_balance:
            peak_balance = account_balance
        dd = (peak_balance - account_balance) / peak_balance
        if dd > max_drawdown:
            max_drawdown = dd

    # ==========================================
    # 4. PRINT MONTH-BY-MONTH BREAKDOWN
    # ==========================================
    print(f"Start Date: {trades[0]['date'].strftime('%Y-%m-%d')} | Starting Balance: ${STARTING_CAPITAL:,.2f}")
    print("-" * 75)
    print(f"{'Month':<15} | {'Trades':<8} | {'End Balance':<15} | {'Monthly ROI':<10}")
    print("-" * 75)
    
    for month, data in monthly_performance.items():
        start_b = data['start_bal']
        end_b = data['end_bal']
        roi = ((end_b - start_b) / start_b) * 100
        print(f"{month:<15} | {data['trades']:<8} | ${end_b:,.2f}{'':<6} | {roi:>6.2f}%")
        
    print("-" * 75)
    
    total_roi = ((account_balance - STARTING_CAPITAL) / STARTING_CAPITAL) * 100
    win_rate = (wins / (wins + losses)) * 100 if (wins + losses) > 0 else 0
    
    print(f"\n📊 FINAL 1-YEAR METRICS:")
    print(f"Final Balance:    ${account_balance:,.2f}")
    print(f"Total Net ROI:    {total_roi:.2f}%")
    print(f"Max Drawdown:     {max_drawdown*100:.2f}%")
    print(f"Total Trades:     {len(trades)} (Wins: {wins}, Losses: {losses}, Breakevens: {breakevens})")
    print(f"True Win Rate:    {win_rate:.2f}% (Excluding Breakevens)")
    print("="*75)

def simulate_trade_with_breakeven(df, start_idx, entry, sl_dist, tp_dist, direction):
    """Simulates trade execution with Institutional Breakeven logic."""
    forward_df = df.iloc[start_idx+1 : start_idx+48] 
    breakeven_trigger_dist = sl_dist / 1.5 
    sl_moved_to_be = False
    
    for _, row in forward_df.iterrows():
        if direction == "BUY":
            if not sl_moved_to_be and row['high'] >= entry + breakeven_trigger_dist:
                sl_moved_to_be = True
            
            if row['low'] <= entry - sl_dist:
                return {'result': 'BREAKEVEN', 'sl_dist': sl_dist, 'tp_dist': tp_dist} if sl_moved_to_be else {'result': 'LOSS', 'sl_dist': sl_dist, 'tp_dist': tp_dist}
                    
            if row['high'] >= entry + tp_dist:
                return {'result': 'WIN', 'sl_dist': sl_dist, 'tp_dist': tp_dist}
                
        else: # SELL
            if not sl_moved_to_be and row['low'] <= entry - breakeven_trigger_dist:
                sl_moved_to_be = True
                
            if row['high'] >= entry + sl_dist:
                return {'result': 'BREAKEVEN', 'sl_dist': sl_dist, 'tp_dist': tp_dist} if sl_moved_to_be else {'result': 'LOSS', 'sl_dist': sl_dist, 'tp_dist': tp_dist}
                    
            if row['low'] <= entry - tp_dist:
                return {'result': 'WIN', 'sl_dist': sl_dist, 'tp_dist': tp_dist}
                
    return {'result': 'BREAKEVEN', 'sl_dist': sl_dist, 'tp_dist': tp_dist} 

if __name__ == "__main__":
    run_funded_scaleup_backtest()

🔄 Connecting to MT5 to fetch 1-Year XAUUSD Data...

📈 THE FUNDED SCALER: 1-YEAR CHRONOLOGICAL COMPOUNDING REPORT
Start Date: 2025-05-15 | Starting Balance: $25,000.00
---------------------------------------------------------------------------
Month           | Trades   | End Balance     | Monthly ROI
---------------------------------------------------------------------------
2025-May        | 7        | $24,931.13       |  -0.28%
2025-Jun        | 8        | $25,300.24       |   1.48%
2025-Jul        | 10       | $24,770.41       |  -2.09%
2025-Aug        | 10       | $25,168.77       |   1.61%
2025-Sep        | 8        | $26,215.34       |   4.16%
2025-Oct        | 17       | $27,034.59       |   3.13%
2025-Nov        | 6        | $26,862.94       |  -0.63%
2025-Dec        | 12       | $27,812.42       |   3.53%
2026-Jan        | 7        | $28,458.67       |   2.32%
2026-Feb        | 8        | $28,414.01       |  -0.16%
2026-Mar        | 9        | $29,147.47       |   2.58%
2026-A

In [8]:
import MetaTrader5 as mt5
import pandas as pd
import pandas_ta as ta
import numpy as np
from datetime import datetime, timedelta
import warnings

warnings.filterwarnings('ignore')

# --- FUNDED ACCOUNT PARAMETERS ---
STARTING_CAPITAL = 25000.0
RISK_PER_TRADE_PCT = 0.005     # 0.5% Risk ($125)
SPREAD_COMMISSION = 5.0        # $5 per lot round trip

def run_asymmetric_backtest():
    print("🔄 Connecting to MT5 to fetch 1-Year XAUUSD Data...")
    if not mt5.initialize():
        print("❌ MT5 Initialization Failed.")
        return
        
    date_to = datetime.now()
    date_from = date_to - timedelta(days=365) 
    
    rates = mt5.copy_rates_range("XAUUSD", mt5.TIMEFRAME_H1, date_from, date_to)
    mt5.shutdown()
    
    if rates is None or len(rates) == 0:
        print("❌ Failed to fetch data.")
        return
        
    df = pd.DataFrame(rates)
    df['time'] = pd.to_datetime(df['time'], unit='s')
    df.set_index('time', inplace=True)
    
    # ==========================================
    # 1. INDICATORS & CHOP FILTER
    # ==========================================
    df['atr'] = ta.atr(df['high'], df['low'], df['close'], length=14)
    df['ema_200'] = ta.ema(df['close'], length=200)
    
    df['sma_20'] = ta.sma(df['close'], length=20)
    df['std_20'] = df['close'].rolling(window=20).std()
    df['z_score'] = (df['close'] - df['sma_20']) / df['std_20']
    
    # THE CHOP FILTER: ADX measures trend strength. > 25 means strong trend.
    adx = ta.adx(df['high'], df['low'], df['close'], length=14)
    if adx is not None:
        df['adx'] = adx['ADX_14']
    else:
        df['adx'] = 0
        
    df.dropna(inplace=True)
    
    # ==========================================
    # 2. EXTRACT HIGH-EXPLOSION TRADES
    # ==========================================
    ALLOWED_HOURS = [13, 14, 15, 16, 17, 18]
    trades = []
    
    for i in range(1, len(df) - 120): # Lookahead 120 hours (5 days) for massive runners
        hour = df.index[i].hour
        
        if hour in ALLOWED_HOURS:
            # We ONLY trade if ADX > 25 (No Chop Allowed)
            if df['adx'].iloc[i] > 25:
                
                # BUY SETUPS
                if df['close'].iloc[i] > df['ema_200'].iloc[i] and df['z_score'].iloc[i] < -1.5:
                    entry = df['close'].iloc[i]
                    sl_dist = df['atr'].iloc[i] * 1.0     # VERY TIGHT LOSS
                    tp1_dist = df['atr'].iloc[i] * 2.0    # 1:2 R:R for the first 50%
                    trail_dist = df['atr'].iloc[i] * 1.5  # Trail the runner
                    
                    result = simulate_runner_trade(df, i, entry, sl_dist, tp1_dist, trail_dist, "BUY")
                    trades.append({'date': df.index[i], 'direction': 'BUY', **result, 'sl_dist': sl_dist})
                        
                # SELL SETUPS
                elif df['close'].iloc[i] < df['ema_200'].iloc[i] and df['z_score'].iloc[i] > 1.5:
                    entry = df['close'].iloc[i]
                    sl_dist = df['atr'].iloc[i] * 1.0     # VERY TIGHT LOSS
                    tp1_dist = df['atr'].iloc[i] * 2.0    # 1:2 R:R for the first 50%
                    trail_dist = df['atr'].iloc[i] * 1.5  # Trail the runner
                    
                    result = simulate_runner_trade(df, i, entry, sl_dist, tp1_dist, trail_dist, "SELL")
                    trades.append({'date': df.index[i], 'direction': 'SELL', **result, 'sl_dist': sl_dist})

    # ==========================================
    # 3. CHRONOLOGICAL COMPOUNDING SIMULATION
    # ==========================================
    print("\n" + "="*75)
    print("🚀 THE ASYMMETRIC SNIPER V8: 1-YEAR RUNNER REPORT")
    print("="*75)
    
    account_balance = STARTING_CAPITAL
    peak_balance = STARTING_CAPITAL
    max_drawdown = 0.0
    
    monthly_performance = {}

    for t in trades:
        month_key = t['date'].strftime('%Y-%b')
        if month_key not in monthly_performance:
            monthly_performance[month_key] = {'start_bal': account_balance, 'end_bal': 0, 'trades': 0}
            
        monthly_performance[month_key]['trades'] += 1
        
        # Risk Calculation
        risk_amount = account_balance * RISK_PER_TRADE_PCT
        sl_dollars = t['sl_dist']
        
        lot_size = round(risk_amount / (sl_dollars * 100), 2)
        if lot_size <= 0: lot_size = 0.01
        
        # Calculate PnL based on the multiplier from the simulator
        # Multiplier of -1 means full loss. Multiplier of 3.5 means huge runner win.
        gross_pnl = risk_amount * t['pnl_multiplier']
        net_pnl = gross_pnl - (SPREAD_COMMISSION * lot_size)
        
        account_balance += net_pnl
        monthly_performance[month_key]['end_bal'] = account_balance
        
        if account_balance > peak_balance:
            peak_balance = account_balance
        dd = (peak_balance - account_balance) / peak_balance
        if dd > max_drawdown:
            max_drawdown = dd

    # ==========================================
    # 4. PRINT MONTH-BY-MONTH BREAKDOWN
    # ==========================================
    if not trades:
        print("❌ No trades triggered. The chop filter might be too strict.")
        return
        
    print(f"Start Date: {trades[0]['date'].strftime('%Y-%m-%d')} | Starting Balance: ${STARTING_CAPITAL:,.2f}")
    print("-" * 75)
    print(f"{'Month':<15} | {'Trades':<8} | {'End Balance':<15} | {'Monthly ROI':<10}")
    print("-" * 75)
    
    for month, data in monthly_performance.items():
        start_b = data['start_bal']
        end_b = data['end_bal']
        roi = ((end_b - start_b) / start_b) * 100
        print(f"{month:<15} | {data['trades']:<8} | ${end_b:,.2f}{'':<6} | {roi:>6.2f}%")
        
    print("-" * 75)
    
    total_roi = ((account_balance - STARTING_CAPITAL) / STARTING_CAPITAL) * 100
    
    print(f"\n📊 FINAL 1-YEAR METRICS (ASYMMETRIC RUNNERS):")
    print(f"Final Balance:    ${account_balance:,.2f}")
    print(f"Total Net ROI:    {total_roi:.2f}%")
    print(f"Max Drawdown:     {max_drawdown*100:.2f}%")
    print(f"Total Trades:     {len(trades)}")
    print("="*75)

def simulate_runner_trade(df, start_idx, entry, sl_dist, tp1_dist, trail_dist, direction):
    """Simulates partial profit taking and trailing stops for massive runners."""
    forward_df = df.iloc[start_idx+1 : start_idx+120] # Give it 5 days to run
    
    hit_tp1 = False
    tp1_price = entry + tp1_dist if direction == "BUY" else entry - tp1_dist
    current_sl = entry - sl_dist if direction == "BUY" else entry + sl_dist
    peak_price = entry
    
    for _, row in forward_df.iterrows():
        if direction == "BUY":
            # 1. Check Stop Loss
            if row['low'] <= current_sl:
                if not hit_tp1:
                    return {'result': 'LOSS', 'pnl_multiplier': -1.0} # Lost 100% of risk
                else:
                    # Stopped out on the runner. 
                    # 50% of position won 2.0R. The other 50% won whatever distance the trailing SL was at.
                    runner_r_multiple = (current_sl - entry) / sl_dist
                    total_r_multiple = (0.5 * 2.0) + (0.5 * runner_r_multiple)
                    return {'result': 'RUNNER_CLOSED', 'pnl_multiplier': total_r_multiple}
            
            # 2. Check TP1 (Secure the bag, move SL to breakeven)
            if not hit_tp1 and row['high'] >= tp1_price:
                hit_tp1 = True
                current_sl = entry # Breakeven
                
            # 3. Update Trailing Stop
            if hit_tp1:
                if row['high'] > peak_price:
                    peak_price = row['high']
                    new_sl = peak_price - trail_dist
                    if new_sl > current_sl:
                        current_sl = new_sl
                        
        else: # SELL
            # 1. Check Stop Loss
            if row['high'] >= current_sl:
                if not hit_tp1:
                    return {'result': 'LOSS', 'pnl_multiplier': -1.0}
                else:
                    runner_r_multiple = (entry - current_sl) / sl_dist
                    total_r_multiple = (0.5 * 2.0) + (0.5 * runner_r_multiple)
                    return {'result': 'RUNNER_CLOSED', 'pnl_multiplier': total_r_multiple}
                    
            # 2. Check TP1
            if not hit_tp1 and row['low'] <= tp1_price:
                hit_tp1 = True
                current_sl = entry
                
            # 3. Update Trailing Stop
            if hit_tp1:
                if row['low'] < peak_price:
                    peak_price = row['low']
                    new_sl = peak_price + trail_dist
                    if new_sl < current_sl:
                        current_sl = new_sl

    # If it runs out of time
    if not hit_tp1:
        return {'result': 'TIME_LIMIT', 'pnl_multiplier': -0.5} 
    else:
        # If the 5 days end and the runner is still alive, we just close it at current profit
        runner_r_multiple = (peak_price - entry) / sl_dist if direction == "BUY" else (entry - peak_price) / sl_dist
        total_r_multiple = (0.5 * 2.0) + (0.5 * runner_r_multiple)
        return {'result': 'TIME_LIMIT', 'pnl_multiplier': total_r_multiple}

if __name__ == "__main__":
    run_asymmetric_backtest()

🔄 Connecting to MT5 to fetch 1-Year XAUUSD Data...

🚀 THE ASYMMETRIC SNIPER V8: 1-YEAR RUNNER REPORT
Start Date: 2025-05-15 | Starting Balance: $25,000.00
---------------------------------------------------------------------------
Month           | Trades   | End Balance     | Monthly ROI
---------------------------------------------------------------------------
2025-May        | 4        | $25,257.45       |   1.03%
2025-Jun        | 4        | $25,383.21       |   0.50%
2025-Jul        | 10       | $25,182.85       |  -0.79%
2025-Aug        | 2        | $25,930.86       |   2.97%
2025-Sep        | 6        | $28,266.69       |   9.01%
2025-Oct        | 5        | $28,437.39       |   0.60%
2025-Nov        | 6        | $27,971.31       |  -1.64%
2025-Dec        | 6        | $28,657.92       |   2.45%
2026-Jan        | 4        | $29,216.23       |   1.95%
2026-Feb        | 1        | $29,490.77       |   0.94%
2026-Mar        | 2        | $29,575.22       |   0.29%
2026-Apr        | 

In [10]:
import MetaTrader5 as mt5
import pandas as pd
import pandas_ta as ta
import numpy as np
import time
from datetime import datetime

# --- ⚙️ EXPERT CONFIGURATION ---
SYMBOL = "XAUUSD"
TIMEFRAME = mt5.TIMEFRAME_H1
MAGIC_NUMBER = 20260508
RISK_PER_TRADE_PCT = 0.005  # 0.5% Risk per trade
ALLOWED_HOURS = [13, 14, 15, 16, 17, 18] # NY Session Power Hours (Broker Time)

print("🔌 INITIALIZING LIVE V8 ASYMMETRIC SNIPER...")
if not mt5.initialize():
    print(f"❌ MT5 Connection Failed. Error: {mt5.last_error()}")
    quit()

account = mt5.account_info()
print(f"✅ Connected: {account.login} | Balance: ${account.balance:,.2f}")
print("---------------------------------------------------------")

def get_market_data():
    """Fetches H1 data and calculates V8 Institutional Indicators"""
    rates = mt5.copy_rates_from_pos(SYMBOL, TIMEFRAME, 0, 250)
    if rates is None: return None
    
    df = pd.DataFrame(rates)
    df['atr'] = ta.atr(df['high'], df['low'], df['close'], length=14)
    df['ema_200'] = ta.ema(df['close'], length=200)
    
    df['sma_20'] = ta.sma(df['close'], length=20)
    df['std_20'] = df['close'].rolling(window=20).std()
    df['z_score'] = (df['close'] - df['sma_20']) / df['std_20']
    
    adx = ta.adx(df['high'], df['low'], df['close'], length=14)
    df['adx'] = adx['ADX_14'] if adx is not None else 0
    
    return df.dropna()

def calculate_lot_size(sl_distance):
    """Calculates dynamic lot size based on 0.5% risk"""
    balance = mt5.account_info().balance
    risk_amount = balance * RISK_PER_TRADE_PCT
    lot_size = risk_amount / (sl_distance * 100) # 1 lot = 100 oz Gold
    
    # MT5 volume rounding
    lot_size = round(lot_size, 2)
    if lot_size < 0.01: lot_size = 0.01
    return lot_size

def execute_trade(direction, entry_price, sl_dist):
    """Executes the initial trade"""
    lot_size = calculate_lot_size(sl_dist)
    
    # We set an initial SL. We DO NOT set a TP, because the bot manages the runner dynamically.
    if direction == "BUY":
        sl_price = entry_price - sl_dist
        order_type = mt5.ORDER_TYPE_BUY
    else:
        sl_price = entry_price + sl_dist
        order_type = mt5.ORDER_TYPE_SELL

    req = {
        "action": mt5.TRADE_ACTION_DEAL,
        "symbol": SYMBOL,
        "volume": float(lot_size),
        "type": order_type,
        "price": entry_price,
        "sl": sl_price,
        "magic": MAGIC_NUMBER,
        "comment": "V8_Sniper_Entry",
        "type_time": mt5.ORDER_TIME_GTC,
        "type_filling": mt5.ORDER_FILLING_IOC,
    }
    
    res = mt5.order_send(req)
    if res.retcode == mt5.TRADE_RETCODE_DONE:
        print(f"🚀 {direction} EXECUTED @ {entry_price:.2f} | Size: {lot_size} | SL: {sl_price:.2f}")
        # Save entry data to a local dictionary for management
        return {"ticket": res.order, "entry": entry_price, "sl_dist": sl_dist, "initial_volume": lot_size, "secured": False}
    else:
        print(f"❌ Execution Failed: {res.comment}")
        return None

def manage_open_positions(active_trade_memory):
    """Handles Partial Close (TP1) and Trailing Stop for the Runner"""
    positions = mt5.positions_get(symbol=SYMBOL)
    if positions is None or len(positions) == 0:
        return {} # No active trades
        
    for pos in positions:
        if pos.magic == MAGIC_NUMBER and pos.ticket in active_trade_memory:
            trade_data = active_trade_memory[pos.ticket]
            entry = trade_data['entry']
            sl_dist = trade_data['sl_dist']
            tp1_price = entry + (sl_dist * 2.0) if pos.type == 0 else entry - (sl_dist * 2.0)
            
            tick = mt5.symbol_info_tick(SYMBOL)
            current_price = tick.bid if pos.type == 0 else tick.ask
            
            # --- PHASE 1: SECURE 50% BAG AT TP1 ---
            if not trade_data['secured']:
                hit_tp1 = (pos.type == 0 and current_price >= tp1_price) or (pos.type == 1 and current_price <= tp1_price)
                
                if hit_tp1:
                    close_vol = round(pos.volume / 2, 2)
                    if close_vol >= 0.01:
                        print(f"💰 TP1 HIT! Securing 50% profits ({close_vol} lots). Moving SL to Breakeven.")
                        
                        # Close half
                        close_req = {
                            "action": mt5.TRADE_ACTION_DEAL,
                            "position": pos.ticket,
                            "symbol": SYMBOL,
                            "volume": close_vol,
                            "type": mt5.ORDER_TYPE_SELL if pos.type == 0 else mt5.ORDER_TYPE_BUY,
                            "price": current_price,
                            "magic": MAGIC_NUMBER,
                            "comment": "V8_TP1_Secure",
                            "type_time": mt5.ORDER_TIME_GTC,
                            "type_filling": mt5.ORDER_FILLING_IOC,
                        }
                        mt5.order_send(close_req)
                        
                        # Move SL to Breakeven
                        sl_req = {
                            "action": mt5.TRADE_ACTION_SLTP,
                            "position": pos.ticket,
                            "sl": entry,
                        }
                        mt5.order_send(sl_req)
                        
                        trade_data['secured'] = True
                        
            # --- PHASE 2: TRAIL THE RUNNER ---
            if trade_data['secured']:
                trail_dist = sl_dist * 1.5
                if pos.type == 0: # BUY RUNNER
                    new_sl = current_price - trail_dist
                    if new_sl > pos.sl and new_sl > entry: # Only trail UP
                        mt5.order_send({"action": mt5.TRADE_ACTION_SLTP, "position": pos.ticket, "sl": new_sl})
                        print(f"📈 Trailing SL Moved UP to {new_sl:.2f}")
                else: # SELL RUNNER
                    new_sl = current_price + trail_dist
                    if new_sl < pos.sl and new_sl < entry: # Only trail DOWN
                        mt5.order_send({"action": mt5.TRADE_ACTION_SLTP, "position": pos.ticket, "sl": new_sl})
                        print(f"📉 Trailing SL Moved DOWN to {new_sl:.2f}")

    return active_trade_memory

# --- 🔁 MAIN LIVE LOOP ---
active_trades = {}
print("⏳ V8 SCANNER ACTIVE. Waiting for Institutional Setups...")

while True:
    try:
        now = datetime.now()
        
        # 1. Trade Management (Runs every 10 seconds)
        active_trades = manage_open_positions(active_trades)
        
        # 2. Setup Scanning (Only check for new trades at the start of a new H1 candle)
        # We check roughly every minute if we are in the power window
        if now.hour in ALLOWED_HOURS and mt5.positions_total() == 0:
            df = get_market_data()
            if df is not None:
                curr = df.iloc[-2] # Last CLOSED candle
                
                # CHOP FILTER
                if curr['adx'] > 25:
                    tick = mt5.symbol_info_tick(SYMBOL)
                    
                    # BUY SIGNAL
                    if curr['close'] > curr['ema_200'] and curr['z_score'] < -1.5:
                        print(f"\n⚡ V8 BUY SIGNAL DETECTED | ADX: {curr['adx']:.1f}")
                        trade_info = execute_trade("BUY", tick.ask, curr['atr'] * 1.0)
                        if trade_info: active_trades[trade_info['ticket']] = trade_info
                            
                    # SELL SIGNAL
                    elif curr['close'] < curr['ema_200'] and curr['z_score'] > 1.5:
                        print(f"\n⚡ V8 SELL SIGNAL DETECTED | ADX: {curr['adx']:.1f}")
                        trade_info = execute_trade("SELL", tick.bid, curr['atr'] * 1.0)
                        if trade_info: active_trades[trade_info['ticket']] = trade_info

        time.sleep(10) # Loop interval

    except Exception as e:
        print(f"⚠️ Live Execution Error: {e}")
        time.sleep(10)

🔌 INITIALIZING LIVE V8 ASYMMETRIC SNIPER...
✅ Connected: 24524229 | Balance: $25,032.90
---------------------------------------------------------
⏳ V8 SCANNER ACTIVE. Waiting for Institutional Setups...



KeyboardInterrupt



In [11]:
import MetaTrader5 as mt5
import pandas as pd
import pandas_ta as ta
import numpy as np
from datetime import datetime, timedelta
import warnings

warnings.filterwarnings('ignore')

# --- THE 10X LEVERAGE PARAMETERS ---
STARTING_CAPITAL = 25000.0
RISK_PER_TRADE_PCT = 0.05      # KAMIKAZE RISK: 5.0% of Account Per Trade
SPREAD_COMMISSION = 5.0        # $5 per lot round trip

def run_apex_compounder():
    print("🚀 INITIALIZING THE APEX COMPOUNDER V10 (MAXIMUM LEVERAGE)...")
    if not mt5.initialize():
        print("❌ MT5 Initialization Failed.")
        return
        
    date_to = datetime.now()
    date_from = date_to - timedelta(days=365) 
    
    rates = mt5.copy_rates_range("XAUUSD", mt5.TIMEFRAME_H1, date_from, date_to)
    mt5.shutdown()
    
    if rates is None or len(rates) == 0:
        print("❌ Failed to fetch data.")
        return
        
    df = pd.DataFrame(rates)
    df['time'] = pd.to_datetime(df['time'], unit='s')
    df.set_index('time', inplace=True)
    
    # ==========================================
    # 1. THE INSTITUTIONAL EDGE
    # ==========================================
    df['atr'] = ta.atr(df['high'], df['low'], df['close'], length=14)
    df['ema_200'] = ta.ema(df['close'], length=200)
    
    df['sma_20'] = ta.sma(df['close'], length=20)
    df['std_20'] = df['close'].rolling(window=20).std()
    df['z_score'] = (df['close'] - df['sma_20']) / df['std_20']
    
    # ADX Filter: Only trade when institutions are moving the market
    adx = ta.adx(df['high'], df['low'], df['close'], length=14)
    df['adx'] = adx['ADX_14'] if adx is not None else 0
        
    df.dropna(inplace=True)
    
    # ==========================================
    # 2. EXTRACT HIGH-EXPLOSION TRADES
    # ==========================================
    ALLOWED_HOURS = [13, 14, 15, 16, 17, 18]
    trades = []
    
    for i in range(1, len(df) - 120): 
        hour = df.index[i].hour
        
        if hour in ALLOWED_HOURS and df['adx'].iloc[i] > 25:
            # BUY SETUPS (Z-Score Sweep)
            if df['close'].iloc[i] > df['ema_200'].iloc[i] and df['z_score'].iloc[i] < -1.5:
                entry = df['close'].iloc[i]
                sl_dist = df['atr'].iloc[i] * 1.0     # Tight 1.0 ATR Stop
                tp1_dist = df['atr'].iloc[i] * 2.0    # TP1 at 2.0 ATR
                trail_dist = df['atr'].iloc[i] * 1.5  # Trail the remaining 50%
                
                result = simulate_apex_runner(df, i, entry, sl_dist, tp1_dist, trail_dist, "BUY")
                trades.append({'date': df.index[i], 'direction': 'BUY', **result, 'sl_dist': sl_dist})
                    
            # SELL SETUPS (Z-Score Sweep)
            elif df['close'].iloc[i] < df['ema_200'].iloc[i] and df['z_score'].iloc[i] > 1.5:
                entry = df['close'].iloc[i]
                sl_dist = df['atr'].iloc[i] * 1.0     
                tp1_dist = df['atr'].iloc[i] * 2.0    
                trail_dist = df['atr'].iloc[i] * 1.5  
                
                result = simulate_apex_runner(df, i, entry, sl_dist, tp1_dist, trail_dist, "SELL")
                trades.append({'date': df.index[i], 'direction': 'SELL', **result, 'sl_dist': sl_dist})

    # ==========================================
    # 3. HYPER-COMPOUNDING SIMULATION
    # ==========================================
    print("\n" + "="*75)
    print(f"🔥 THE APEX COMPOUNDER: 1-YEAR ROAD TO 10X")
    print(f"Risk Per Trade: {RISK_PER_TRADE_PCT*100}% | Starting Capital: ${STARTING_CAPITAL:,.2f}")
    print("="*75)
    
    account_balance = STARTING_CAPITAL
    peak_balance = STARTING_CAPITAL
    max_drawdown = 0.0
    
    monthly_performance = {}

    for t in trades:
        month_key = t['date'].strftime('%Y-%b')
        if month_key not in monthly_performance:
            monthly_performance[month_key] = {'start_bal': account_balance, 'end_bal': 0, 'trades': 0}
            
        monthly_performance[month_key]['trades'] += 1
        
        # AGGRESSIVE RISK SCALING
        risk_amount = account_balance * RISK_PER_TRADE_PCT
        sl_dollars = t['sl_dist']
        
        lot_size = round(risk_amount / (sl_dollars * 100), 2)
        if lot_size <= 0: lot_size = 0.01
        
        gross_pnl = risk_amount * t['pnl_multiplier']
        net_pnl = gross_pnl - (SPREAD_COMMISSION * lot_size)
        
        account_balance += net_pnl
        monthly_performance[month_key]['end_bal'] = account_balance
        
        if account_balance > peak_balance:
            peak_balance = account_balance
        dd = (peak_balance - account_balance) / peak_balance
        if dd > max_drawdown:
            max_drawdown = dd

        # Margin Call Protection (If account drops below 10% of starting capital)
        if account_balance < (STARTING_CAPITAL * 0.1):
            print("\n💀 MARGIN CALL: Account blown. Leverage was too high for the drawdown sequence.")
            return

    # ==========================================
    # 4. PRINT MONTH-BY-MONTH BREAKDOWN
    # ==========================================
    if not trades:
        print("❌ No trades triggered.")
        return
        
    print("-" * 75)
    print(f"{'Month':<15} | {'Trades':<8} | {'End Balance':<15} | {'Monthly Growth':<10}")
    print("-" * 75)
    
    for month, data in monthly_performance.items():
        start_b = data['start_bal']
        end_b = data['end_bal']
        roi = ((end_b - start_b) / start_b) * 100
        print(f"{month:<15} | {data['trades']:<8} | ${end_b:,.2f}{'':<6} | {roi:>6.2f}%")
        
    print("-" * 75)
    
    total_roi = ((account_balance - STARTING_CAPITAL) / STARTING_CAPITAL) * 100
    multiplier = account_balance / STARTING_CAPITAL
    
    print(f"\n📈 THE FINAL 10X METRICS:")
    print(f"Final Balance:    ${account_balance:,.2f}")
    print(f"Capital Multiple: {multiplier:.2f}x")
    print(f"Total Net ROI:    {total_roi:,.2f}%")
    print(f"Max Drawdown:     {max_drawdown*100:.2f}% (Expect 30%+ with 5% risk)")
    print(f"Total Trades:     {len(trades)}")
    print("="*75)

def simulate_apex_runner(df, start_idx, entry, sl_dist, tp1_dist, trail_dist, direction):
    """Simulates the Asymmetric Runner for maximum multiplier extraction."""
    forward_df = df.iloc[start_idx+1 : start_idx+120] 
    
    hit_tp1 = False
    tp1_price = entry + tp1_dist if direction == "BUY" else entry - tp1_dist
    current_sl = entry - sl_dist if direction == "BUY" else entry + sl_dist
    peak_price = entry
    
    for _, row in forward_df.iterrows():
        if direction == "BUY":
            if row['low'] <= current_sl:
                if not hit_tp1:
                    return {'result': 'LOSS', 'pnl_multiplier': -1.0} 
                else:
                    runner_r_multiple = (current_sl - entry) / sl_dist
                    total_r_multiple = (0.5 * 2.0) + (0.5 * runner_r_multiple)
                    return {'result': 'RUNNER_CLOSED', 'pnl_multiplier': total_r_multiple}
            
            if not hit_tp1 and row['high'] >= tp1_price:
                hit_tp1 = True
                current_sl = entry 
                
            if hit_tp1:
                if row['high'] > peak_price:
                    peak_price = row['high']
                    new_sl = peak_price - trail_dist
                    if new_sl > current_sl:
                        current_sl = new_sl
                        
        else: # SELL
            if row['high'] >= current_sl:
                if not hit_tp1:
                    return {'result': 'LOSS', 'pnl_multiplier': -1.0}
                else:
                    runner_r_multiple = (entry - current_sl) / sl_dist
                    total_r_multiple = (0.5 * 2.0) + (0.5 * runner_r_multiple)
                    return {'result': 'RUNNER_CLOSED', 'pnl_multiplier': total_r_multiple}
                    
            if not hit_tp1 and row['low'] <= tp1_price:
                hit_tp1 = True
                current_sl = entry
                
            if hit_tp1:
                if row['low'] < peak_price:
                    peak_price = row['low']
                    new_sl = peak_price + trail_dist
                    if new_sl < current_sl:
                        current_sl = new_sl

    if not hit_tp1:
        return {'result': 'TIME_LIMIT', 'pnl_multiplier': -0.5} 
    else:
        runner_r_multiple = (peak_price - entry) / sl_dist if direction == "BUY" else (entry - peak_price) / sl_dist
        total_r_multiple = (0.5 * 2.0) + (0.5 * runner_r_multiple)
        return {'result': 'TIME_LIMIT', 'pnl_multiplier': total_r_multiple}

if __name__ == "__main__":
    run_apex_compounder()

🚀 INITIALIZING THE APEX COMPOUNDER V10 (MAXIMUM LEVERAGE)...

🔥 THE APEX COMPOUNDER: 1-YEAR ROAD TO 10X
Risk Per Trade: 5.0% | Starting Capital: $25,000.00
---------------------------------------------------------------------------
Month           | Trades   | End Balance     | Monthly Growth
---------------------------------------------------------------------------
2025-May        | 4        | $27,365.46       |   9.46%
2025-Jun        | 4        | $28,552.06       |   4.34%
2025-Jul        | 10       | $25,862.32       |  -9.42%
2025-Aug        | 2        | $34,049.98       |  31.66%
2025-Sep        | 6        | $74,703.90       | 119.39%
2025-Oct        | 5        | $78,322.47       |   4.84%
2025-Nov        | 6        | $65,795.68       | -15.99%
2025-Dec        | 6        | $82,598.21       |  25.54%
2026-Jan        | 4        | $99,188.87       |  20.09%
2026-Feb        | 1        | $108,510.02       |   9.40%
2026-Mar        | 2        | $111,230.28       |   2.51%
2026-Apr    

In [12]:
import MetaTrader5 as mt5
import pandas as pd
import pandas_ta as ta
import numpy as np
from datetime import datetime, timedelta
import warnings

warnings.filterwarnings('ignore')

# --- THE XM BONUS LEVERAGE PARAMETERS ---
STARTING_CASH = 500.0          # Your actual deposit
BONUS_CREDIT = 250.0           # XM 50% Bonus (Not withdrawable, used for margin)
RISK_PER_TRADE_PCT = 0.05      # 5.0% Risk of EQUITY
SPREAD_COMMISSION = 5.0        # $5 per lot round trip

def run_xm_bonus_backtest():
    print("🚀 INITIALIZING XM BONUS BACKTESTER ($500 Cash + $250 Credit)...")
    if not mt5.initialize():
        print("❌ MT5 Initialization Failed.")
        return
        
    date_to = datetime.now()
    date_from = date_to - timedelta(days=365) 
    
    rates = mt5.copy_rates_range("XAUUSD", mt5.TIMEFRAME_H1, date_from, date_to)
    mt5.shutdown()
    
    if rates is None or len(rates) == 0:
        print("❌ Failed to fetch data.")
        return
        
    df = pd.DataFrame(rates)
    df['time'] = pd.to_datetime(df['time'], unit='s')
    df.set_index('time', inplace=True)
    
    # ==========================================
    # 1. THE INSTITUTIONAL EDGE (V10 Logic)
    # ==========================================
    df['atr'] = ta.atr(df['high'], df['low'], df['close'], length=14)
    df['ema_200'] = ta.ema(df['close'], length=200)
    
    df['sma_20'] = ta.sma(df['close'], length=20)
    df['std_20'] = df['close'].rolling(window=20).std()
    df['z_score'] = (df['close'] - df['sma_20']) / df['std_20']
    
    adx = ta.adx(df['high'], df['low'], df['close'], length=14)
    df['adx'] = adx['ADX_14'] if adx is not None else 0
        
    df.dropna(inplace=True)
    
    # ==========================================
    # 2. EXTRACT HIGH-EXPLOSION TRADES
    # ==========================================
    ALLOWED_HOURS = [13, 14, 15, 16, 17, 18]
    trades = []
    
    for i in range(1, len(df) - 120): 
        hour = df.index[i].hour
        
        if hour in ALLOWED_HOURS and df['adx'].iloc[i] > 25:
            # BUY SETUPS 
            if df['close'].iloc[i] > df['ema_200'].iloc[i] and df['z_score'].iloc[i] < -1.5:
                entry = df['close'].iloc[i]
                sl_dist = df['atr'].iloc[i] * 1.0     
                tp1_dist = df['atr'].iloc[i] * 2.0    
                trail_dist = df['atr'].iloc[i] * 1.5  
                
                result = simulate_apex_runner(df, i, entry, sl_dist, tp1_dist, trail_dist, "BUY")
                trades.append({'date': df.index[i], 'direction': 'BUY', **result, 'sl_dist': sl_dist})
                    
            # SELL SETUPS
            elif df['close'].iloc[i] < df['ema_200'].iloc[i] and df['z_score'].iloc[i] > 1.5:
                entry = df['close'].iloc[i]
                sl_dist = df['atr'].iloc[i] * 1.0     
                tp1_dist = df['atr'].iloc[i] * 2.0    
                trail_dist = df['atr'].iloc[i] * 1.5  
                
                result = simulate_apex_runner(df, i, entry, sl_dist, tp1_dist, trail_dist, "SELL")
                trades.append({'date': df.index[i], 'direction': 'SELL', **result, 'sl_dist': sl_dist})

    # ==========================================
    # 3. HYPER-COMPOUNDING SIMULATION (XM MATH)
    # ==========================================
    print("\n" + "="*75)
    print(f"🔥 XM BONUS COMPOUNDER: THE $500 INCUBATOR")
    print(f"Starting Equity: ${(STARTING_CASH + BONUS_CREDIT):.2f} | Risk: {RISK_PER_TRADE_PCT*100}% of Equity")
    print("="*75)
    
    equity = STARTING_CASH + BONUS_CREDIT
    peak_equity = equity
    max_drawdown = 0.0
    
    monthly_performance = {}
    blown_account = False

    for t in trades:
        month_key = t['date'].strftime('%Y-%b')
        if month_key not in monthly_performance:
            monthly_performance[month_key] = {'start_equity': equity, 'end_equity': 0, 'trades': 0}
            
        monthly_performance[month_key]['trades'] += 1
        
        # Risk calculated on total EQUITY
        risk_amount = equity * RISK_PER_TRADE_PCT
        sl_dollars = t['sl_dist']
        
        # Strict Micro-Lot rounding (must be at least 0.02 so we can split it at TP1 to 0.01)
        lot_size = risk_amount / (sl_dollars * 100)
        lot_size = max(0.02, round(lot_size, 2))
        
        # Calculate PnL
        gross_pnl = risk_amount * t['pnl_multiplier']
        net_pnl = gross_pnl - (SPREAD_COMMISSION * lot_size)
        
        equity += net_pnl
        monthly_performance[month_key]['end_equity'] = equity
        
        # Drawdown tracking
        if equity > peak_equity:
            peak_equity = equity
        dd = (peak_equity - equity) / peak_equity
        if dd > max_drawdown:
            max_drawdown = dd

        # MARGIN CALL CHECK: Did we lose our actual $500 cash?
        real_cash_left = equity - BONUS_CREDIT
        if real_cash_left <= 0:
            print(f"\n💀 MARGIN CALL ({t['date'].strftime('%Y-%m-%d')}): You lost the $500 cash. Bonus does not save you from a stop-out.")
            blown_account = True
            break

    # ==========================================
    # 4. PRINT RESULTS
    # ==========================================
    if not trades:
        print("❌ No trades triggered.")
        return
        
    print("-" * 75)
    print(f"{'Month':<15} | {'Trades':<8} | {'End Equity':<15} | {'Real Cash':<10}")
    print("-" * 75)
    
    for month, data in monthly_performance.items():
        if data['end_equity'] > 0:
            real_cash = data['end_equity'] - BONUS_CREDIT
            print(f"{month:<15} | {data['trades']:<8} | ${data['end_equity']:,.2f}{'':<5} | ${real_cash:,.2f}")
        
    print("-" * 75)
    
    if blown_account:
        print(f"\n❌ FINAL VERDICT: Account Blown. Max Drawdown Hit: {max_drawdown*100:.2f}%")
    else:
        final_cash = equity - BONUS_CREDIT
        net_profit = final_cash - STARTING_CASH
        total_roi = (net_profit / STARTING_CASH) * 100
        
        print(f"\n💰 THE INCUBATOR RESULTS:")
        print(f"Final Total Equity: ${equity:,.2f}")
        print(f"Final REAL Cash:    ${final_cash:,.2f} (Withdrawable)")
        print(f"Net Profit:         ${net_profit:,.2f}")
        print(f"Return on Deposit:  {total_roi:,.2f}%")
        print(f"Max Drawdown:       {max_drawdown*100:.2f}%")
        print(f"Total Trades:       {len(trades)}")
        print("="*75)

def simulate_apex_runner(df, start_idx, entry, sl_dist, tp1_dist, trail_dist, direction):
    """Simulates the Asymmetric Runner for maximum multiplier extraction."""
    forward_df = df.iloc[start_idx+1 : start_idx+120] 
    
    hit_tp1 = False
    tp1_price = entry + tp1_dist if direction == "BUY" else entry - tp1_dist
    current_sl = entry - sl_dist if direction == "BUY" else entry + sl_dist
    peak_price = entry
    
    for _, row in forward_df.iterrows():
        if direction == "BUY":
            if row['low'] <= current_sl:
                if not hit_tp1: return {'result': 'LOSS', 'pnl_multiplier': -1.0} 
                else: return {'result': 'RUNNER_CLOSED', 'pnl_multiplier': (0.5 * 2.0) + (0.5 * ((current_sl - entry) / sl_dist))}
            if not hit_tp1 and row['high'] >= tp1_price:
                hit_tp1 = True
                current_sl = entry 
            if hit_tp1 and row['high'] > peak_price:
                peak_price = row['high']
                new_sl = peak_price - trail_dist
                if new_sl > current_sl: current_sl = new_sl
                        
        else: # SELL
            if row['high'] >= current_sl:
                if not hit_tp1: return {'result': 'LOSS', 'pnl_multiplier': -1.0}
                else: return {'result': 'RUNNER_CLOSED', 'pnl_multiplier': (0.5 * 2.0) + (0.5 * ((entry - current_sl) / sl_dist))}
            if not hit_tp1 and row['low'] <= tp1_price:
                hit_tp1 = True
                current_sl = entry
            if hit_tp1 and row['low'] < peak_price:
                peak_price = row['low']
                new_sl = peak_price + trail_dist
                if new_sl < current_sl: current_sl = new_sl

    if not hit_tp1: return {'result': 'TIME_LIMIT', 'pnl_multiplier': -0.5} 
    else: return {'result': 'TIME_LIMIT', 'pnl_multiplier': (0.5 * 2.0) + (0.5 * (abs(peak_price - entry) / sl_dist))}

if __name__ == "__main__":
    run_xm_bonus_backtest()

🚀 INITIALIZING XM BONUS BACKTESTER ($500 Cash + $250 Credit)...

🔥 XM BONUS COMPOUNDER: THE $500 INCUBATOR
Starting Equity: $750.00 | Risk: 5.0% of Equity
---------------------------------------------------------------------------
Month           | Trades   | End Equity      | Real Cash 
---------------------------------------------------------------------------
2025-May        | 4        | $820.98      | $570.98
2025-Jun        | 4        | $856.59      | $606.59
2025-Jul        | 10       | $775.90      | $525.90
2025-Aug        | 2        | $1,021.59      | $771.59
2025-Sep        | 6        | $2,241.30      | $1,991.30
2025-Oct        | 5        | $2,349.85      | $2,099.85
2025-Nov        | 6        | $1,973.97      | $1,723.97
2025-Dec        | 6        | $2,478.09      | $2,228.09
2026-Jan        | 4        | $2,975.79      | $2,725.79
2026-Feb        | 1        | $3,255.43      | $3,005.43
2026-Mar        | 2        | $3,337.05      | $3,087.05
2026-Apr        | 1        | $3,5

In [2]:
import MetaTrader5 as mt5
import pandas as pd
import pandas_ta as ta
import numpy as np
import time
from datetime import datetime

# --- ⚙️ LIVE APEX V10 CONFIGURATION ---
SYMBOL = "XAUUSD"
TIMEFRAME = mt5.TIMEFRAME_H1
MAGIC_NUMBER = 20261010
RISK_PER_TRADE_PCT = 0.05  # KAMIKAZE RISK: 5.0% of Balance
ALLOWED_HOURS = [13, 14, 15, 16, 17, 18] # NY Session Power Hours

print("🚀 INITIALIZING THE LIVE APEX COMPOUNDER V10...")
if not mt5.initialize():
    print(f"❌ MT5 Connection Failed. Error: {mt5.last_error()}")
    quit()

account = mt5.account_info()
print(f"✅ Connected: {account.login} | Balance: ${account.balance:,.2f}")
print(f"⚠️ Risk Level: 5.0% (${account.balance * RISK_PER_TRADE_PCT:.2f} per trade)")
print("---------------------------------------------------------")

def get_market_data():
    """Fetches H1 data and calculates Institutional Z-Score / ADX"""
    rates = mt5.copy_rates_from_pos(SYMBOL, TIMEFRAME, 0, 250)
    if rates is None: return None
    
    df = pd.DataFrame(rates)
    df['atr'] = ta.atr(df['high'], df['low'], df['close'], length=14)
    df['ema_200'] = ta.ema(df['close'], length=200)
    
    df['sma_20'] = ta.sma(df['close'], length=20)
    df['std_20'] = df['close'].rolling(window=20).std()
    df['z_score'] = (df['close'] - df['sma_20']) / df['std_20']
    
    adx = ta.adx(df['high'], df['low'], df['close'], length=14)
    df['adx'] = adx['ADX_14'] if adx is not None else 0
    
    return df.dropna()

def calculate_lot_size(sl_distance):
    """Calculates dynamic lot size based on strict 5% risk"""
    balance = mt5.account_info().balance
    risk_amount = balance * RISK_PER_TRADE_PCT
    lot_size = risk_amount / (sl_distance * 100) # 1 lot = 100 oz Gold
    
    # MT5 volume rounding for Micro/Standard accounts
    lot_size = max(0.02, round(lot_size, 2)) # Min 0.02 so we can split it at TP1
    return lot_size

def execute_trade(direction, entry_price, sl_dist):
    """Executes the initial high-leverage trade"""
    lot_size = calculate_lot_size(sl_dist)
    
    # We do NOT set a hard TP in MT5. The script manages the runner.
    if direction == "BUY":
        sl_price = entry_price - sl_dist
        order_type = mt5.ORDER_TYPE_BUY
    else:
        sl_price = entry_price + sl_dist
        order_type = mt5.ORDER_TYPE_SELL

    req = {
        "action": mt5.TRADE_ACTION_DEAL,
        "symbol": SYMBOL,
        "volume": float(lot_size),
        "type": order_type,
        "price": entry_price,
        "sl": sl_price,
        "magic": MAGIC_NUMBER,
        "comment": "Apex_V10_Entry",
        "type_time": mt5.ORDER_TIME_GTC,
        "type_filling": mt5.ORDER_FILLING_IOC,
    }
    
    res = mt5.order_send(req)
    if res.retcode == mt5.TRADE_RETCODE_DONE:
        print(f"🔥 {direction} EXECUTED @ {entry_price:.2f} | Size: {lot_size} | SL: {sl_price:.2f}")
        return {"ticket": res.order, "entry": entry_price, "sl_dist": sl_dist, "secured": False}
    else:
        print(f"❌ Execution Failed: {res.comment}")
        return None

def manage_open_positions(active_trade_memory):
    """Handles the 50% TP1 Bag Secure and the Trailing Runner"""
    positions = mt5.positions_get(symbol=SYMBOL)
    if positions is None or len(positions) == 0:
        return {} 
        
    for pos in positions:
        if pos.magic == MAGIC_NUMBER and pos.ticket in active_trade_memory:
            trade_data = active_trade_memory[pos.ticket]
            entry = trade_data['entry']
            sl_dist = trade_data['sl_dist']
            tp1_price = entry + (sl_dist * 2.0) if pos.type == 0 else entry - (sl_dist * 2.0)
            
            tick = mt5.symbol_info_tick(SYMBOL)
            current_price = tick.bid if pos.type == 0 else tick.ask
            
            # --- PHASE 1: SECURE 50% BAG AT TP1 ---
            if not trade_data['secured']:
                hit_tp1 = (pos.type == 0 and current_price >= tp1_price) or (pos.type == 1 and current_price <= tp1_price)
                
                if hit_tp1:
                    close_vol = max(0.01, round(pos.volume / 2, 2))
                    if close_vol > 0:
                        print(f"💰 TP1 HIT! Securing 50% profits ({close_vol} lots). Moving SL to Breakeven.")
                        
                        # Close half
                        close_req = {
                            "action": mt5.TRADE_ACTION_DEAL,
                            "position": pos.ticket,
                            "symbol": SYMBOL,
                            "volume": close_vol,
                            "type": mt5.ORDER_TYPE_SELL if pos.type == 0 else mt5.ORDER_TYPE_BUY,
                            "price": current_price,
                            "magic": MAGIC_NUMBER,
                            "comment": "Apex_TP1_Secure",
                            "type_time": mt5.ORDER_TIME_GTC,
                            "type_filling": mt5.ORDER_FILLING_IOC,
                        }
                        mt5.order_send(close_req)
                        
                        # Move SL to Breakeven
                        sl_req = {
                            "action": mt5.TRADE_ACTION_SLTP,
                            "position": pos.ticket,
                            "sl": entry,
                        }
                        mt5.order_send(sl_req)
                        
                        trade_data['secured'] = True
                        
            # --- PHASE 2: TRAIL THE RUNNER (1.5x ATR) ---
            if trade_data['secured']:
                trail_dist = sl_dist * 1.5
                if pos.type == 0: # BUY RUNNER
                    new_sl = current_price - trail_dist
                    if new_sl > pos.sl and new_sl > entry: 
                        mt5.order_send({"action": mt5.TRADE_ACTION_SLTP, "position": pos.ticket, "sl": new_sl})
                        print(f"📈 Trailing SL Moved UP to {new_sl:.2f}")
                else: # SELL RUNNER
                    new_sl = current_price + trail_dist
                    if new_sl < pos.sl and new_sl < entry: 
                        mt5.order_send({"action": mt5.TRADE_ACTION_SLTP, "position": pos.ticket, "sl": new_sl})
                        print(f"📉 Trailing SL Moved DOWN to {new_sl:.2f}")

    return active_trade_memory

# --- 🔁 MAIN LIVE LOOP ---
active_trades = {}
print("⏳ APEX SCANNER ACTIVE. Waiting for Institutional Setups...")

while True:
    try:
        now = datetime.now()
        
        # 1. Trade Management (Runs every 5 seconds for precise trailing stops)
        active_trades = manage_open_positions(active_trades)
        
        # 2. Setup Scanning (Only check for new trades at the start of a new H1 candle)
        if now.hour in ALLOWED_HOURS and mt5.positions_total() == 0:
            df = get_market_data()
            if df is not None:
                curr = df.iloc[-2] # Last CLOSED candle
                
                # CHOP FILTER (ADX > 25)
                if curr['adx'] > 25:
                    tick = mt5.symbol_info_tick(SYMBOL)
                    
                    # BUY SIGNAL (Macro UP, Z-Score Sweep DOWN)
                    if curr['close'] > curr['ema_200'] and curr['z_score'] < -1.5:
                        print(f"\n⚡ APEX BUY SIGNAL | ADX: {curr['adx']:.1f}")
                        trade_info = execute_trade("BUY", tick.ask, curr['atr'] * 1.0)
                        if trade_info: active_trades[trade_info['ticket']] = trade_info
                            
                    # SELL SIGNAL (Macro DOWN, Z-Score Sweep UP)
                    elif curr['close'] < curr['ema_200'] and curr['z_score'] > 1.5:
                        print(f"\n⚡ APEX SELL SIGNAL | ADX: {curr['adx']:.1f}")
                        trade_info = execute_trade("SELL", tick.bid, curr['atr'] * 1.0)
                        if trade_info: active_trades[trade_info['ticket']] = trade_info

        time.sleep(5) 

    except Exception as e:
        print(f"⚠️ Live Execution Error: {e}")
        time.sleep(5)

🚀 INITIALIZING THE LIVE APEX COMPOUNDER V10...
✅ Connected: 345148129 | Balance: $1,000.00
⚠️ Risk Level: 5.0% ($50.00 per trade)
---------------------------------------------------------
⏳ APEX SCANNER ACTIVE. Waiting for Institutional Setups...



KeyboardInterrupt



In [ ]:
import MetaTrader5 as mt5
import time
import pandas as pd
import numpy as np

# --- CONFIGURATION ---
SYMBOL = "GOLD"
TIMEFRAME = mt5.TIMEFRAME_H1
RISK_PCT = 0.005  # 0.5% Micro-Risk
MAGIC_NUMBER = 123456  # Unique ID for this bot's trades

def get_data(n=100):
    rates = mt5.copy_rates_from_pos(SYMBOL, TIMEFRAME, 0, n)
    df = pd.DataFrame(rates)
    df['time'] = pd.to_datetime(df['time'], unit='s')
    return df

def place_order(type, price, sl, tp, volume):
    request = {
        "action": mt5.TRADE_ACTION_DEAL,
        "symbol": SYMBOL,
        "volume": volume,
        "type": type,
        "price": price,
        "sl": sl,
        "tp": tp,
        "magic": MAGIC_NUMBER,
        "comment": "Z-Score Hybrid Bot",
        "type_time": mt5.ORDER_TIME_GTC,
        "type_filling": mt5.ORDER_FILLING_IOC,
    }
    result = mt5.order_send(request)
    return result

def run_bot():
    print(f"Bot started. Monitoring {SYMBOL}...")
    while True:
        # 1. Check if we already have an open trade
        if mt5.positions_get(symbol=SYMBOL, magic=MAGIC_NUMBER):
            time.sleep(60)
            continue

        # 2. Check Time Filter (13:00 - 19:00 Broker Time)
        current_hour = time.strftime('%H', time.gmtime()) # Adjust for Broker GMT
        if not (13 <= int(current_hour) <= 19):
            time.sleep(60)
            continue

        # 3. Calculate Indicators
        df = get_data(60)
        df['ATR'] = (df['high'] - df['low']).rolling(window=20).mean()
        df['SMA'] = df['close'].rolling(window=50).mean()
        df['Z'] = (df['close'] - df['SMA']) / df['close'].rolling(window=50).std()
        
        last_z = df['Z'].iloc[-1]
        last_atr = df['ATR'].iloc[-1]
        price = mt5.symbol_info_tick(SYMBOL).ask if last_z > 1.5 else mt5.symbol_info_tick(SYMBOL).bid

        # 4. Logic Execution
        if abs(last_z) > 1.5:
            # Volume calculation (simplified - check broker contract size)
            balance = mt5.account_info().balance
            risk_amount = balance * RISK_PCT
            stop_dist = last_atr * 1.0
            
            # Standard Gold lot calculation: Risk / (StopDist * TickValue)
            volume = round(risk_amount / (stop_dist * 100), 2) 
            volume = max(0.01, volume) # Ensure minimum lot size

            if last_z > 1.5: # BUY
                sl = price - stop_dist
                tp = price + (stop_dist * 2.0)
                place_order(mt5.ORDER_TYPE_BUY, price, sl, tp, volume)
                print(f"BUY Order Placed: Z={last_z:.2f}")
            
            elif last_z < -1.5: # SELL
                sl = price + stop_dist
                tp = price - (stop_dist * 2.0)
                place_order(mt5.ORDER_TYPE_SELL, price, sl, tp, volume)
                print(f"SELL Order Placed: Z={last_z:.2f}")

        time.sleep(60) # Wait 1 minute before next check

if __name__ == "__main__":
    if mt5.initialize():
        run_bot()

Bot started. Monitoring GOLD...
